Check dataset class distribution (so we can confirm if class 4 is underrepresented).

Identify which classes are most often confused with class 4 — from confusion matrix data.

Prepare class weighting for future fine-tuning — so if we fine-tune v9 later, class 4 will be given more importance without affecting well-performing classes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import importlib.util
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# ===============================
# ✅ Load MicroExpressionDataset from load_sequences.py
# ===============================
load_seq_path = "/content/drive/MyDrive/deepfake-detection/microexpression/load_sequences.py"

if not os.path.exists(load_seq_path):
    raise FileNotFoundError(f"❌ load_sequences.py not found at {load_seq_path}")

spec = importlib.util.spec_from_file_location("load_sequences", load_seq_path)
load_sequences = importlib.util.module_from_spec(spec)
spec.loader.exec_module(load_sequences)

MicroExpressionDataset = load_sequences.MicroExpressionDataset
print("📂 MicroExpressionDataset imported successfully!")

# ===============================
# Config Paths
# ===============================
root_dir = "/content/drive/MyDrive/deepfake-detection/dataset/microexpression_processed"
cache_dir = "/content/drive/MyDrive/deepfake-detection/microexpression/cache"
model_path = "/content/drive/MyDrive/deepfake-detection/microexpression/models/resnet18_microexpr_v9_best.pth"
reports_dir = "/content/drive/MyDrive/deepfake-detection/microexpression/reports"

os.makedirs(reports_dir, exist_ok=True)

batch_size = 8
seq_len = 16
num_classes = 6  # emotions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# Dataset & DataLoader
# ===============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])

dataset = MicroExpressionDataset(root_dir=root_dir, seq_len=seq_len,
                                 transform=transform, cache_dir=cache_dir)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"✅ Dataset loaded. Total samples: {len(dataset)}")

# ===============================
# Model Definition
# ===============================
class MicroExpressionNet(nn.Module):
    def __init__(self, num_classes):
        super(MicroExpressionNet, self).__init__()
        resnet = models.resnet18(pretrained=False)
        resnet.fc = nn.Identity()
        self.frame_branch = resnet

        self.au_branch = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 128),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(512 + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, frames, aus):
        B, T, C, H, W = frames.size()
        frames = frames.view(B * T, C, H, W)
        frame_features = self.frame_branch(frames)
        frame_features = frame_features.view(B, T, -1).mean(dim=1)

        aus_features = aus.view(B * T, -1)
        aus_features = self.au_branch(aus_features)
        aus_features = aus_features.view(B, T, -1).mean(dim=1)

        combined = torch.cat([frame_features, aus_features], dim=1)
        out = self.fc(combined)
        return out

# ===============================
# Load Model
# ===============================
model = MicroExpressionNet(num_classes=num_classes).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f"✅ Loaded model from {model_path}")

# ===============================
# Evaluation
# ===============================
all_labels = []
all_preds = []
misclassified_samples = []

with torch.no_grad():
    for idx, (frames, aus, labels) in enumerate(loader):
        frames, aus, labels = frames.to(device), aus.to(device), labels.to(device)
        outputs = model(frames, aus)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

        # Track misclassified
        for i in range(len(labels)):
            if preds[i] != labels[i]:
                misclassified_samples.append({
                    "index": idx * batch_size + i,
                    "true_label": labels[i].item(),
                    "pred_label": preds[i].item()
                })

# ===============================
# Classification Report
# ===============================
report = classification_report(all_labels, all_preds, digits=4)
print("\n📊 Classification Report:\n", report)

report_path = os.path.join(reports_dir, "classification_report_v9.txt")
with open(report_path, "w") as f:
    f.write(report)
print(f"📄 Classification report saved to: {report_path}")

# ===============================
# Confusion Matrix
# ===============================
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f"Class {i}" for i in range(num_classes)],
            yticklabels=[f"Class {i}" for i in range(num_classes)])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - MicroExpressionNet v9")
cm_path = os.path.join(reports_dir, "confusion_matrix_v9.png")
plt.savefig(cm_path)
plt.close()
print(f"📊 Confusion matrix saved to: {cm_path}")

# ===============================
# Save Misclassified Samples
# ===============================
misclassified_path = os.path.join(reports_dir, "misclassified_samples_v9.txt")
with open(misclassified_path, "w") as f:
    for item in misclassified_samples:
        f.write(f"Index: {item['index']}, True: {item['true_label']}, Pred: {item['pred_label']}\n")
print(f"⚠️ Misclassified samples saved to: {misclassified_path}")

# ===============================
# Per-class sample counts
# ===============================
counts = Counter(all_labels)
print("\n📌 Sample count per class:")
for cls, count in counts.items():
    print(f"Class {cls}: {count} samples")

# ===============================
# Identify worst performing classes
# ===============================
class_accuracy = cm.diagonal() / cm.sum(axis=1)
worst_classes = np.argsort(class_accuracy)[:3]  # 3 worst classes
print("\n⚠️ Worst performing classes:")
for cls in worst_classes:
    print(f"Class {cls} - Accuracy: {class_accuracy[cls]:.2f}")

print("\n✅ Evaluation complete.")


📂 MicroExpressionDataset imported successfully!
📂 Initializing MicroExpressionDataset...
🗂 Cache directory set to: /content/drive/MyDrive/deepfake-detection/microexpression/cache
🔍 Scanning dataset folders...
✅ Found 220 video samples across 6 emotion categories.
✅ Dataset loaded. Total samples: 220


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Loaded model from /content/drive/MyDrive/deepfake-detection/microexpression/models/resnet18_microexpr_v9_best.pth
⚡ Loading from cache: EP03_04.pt⚡ Loading from cache: EP02_01f.pt

⚡ Loading from cache: EP09_01.pt
⚡ Loading from cache: EP08_07.pt
⚡ Loading from cache: EP09_10.pt
⚡ Loading from cache: EP03_01.pt
⚡ Loading from cache: EP09_03.pt
⚡ Loading from cache: EP01_01.pt
⚡ Loading from cache: EP09_04.pt
⚡ Loading from cache: EP05_05.pt
⚡ Loading from cache: EP09_06.pt
⚡ Loading from cache: EP06_02f.pt
⚡ Loading from cache: EP03_02.pt
⚡ Loading from cache: EP09f.pt
⚡ Loading from cache: EP01_05.pt
⚡ Loading from cache: EP15_05.pt
⚡ Loading from cache: EP01_01f.pt
⚡ Loading from cache: EP04_02f.pt
⚡ Loading from cache: EP01_02f.pt
⚡ Loading from cache: EP01_06.pt
⚡ Loading from cache: EP08_02.pt
⚡ Loading from cache: EP01_15.pt
⚡ Loading from cache: EP02_01.pt
⚡ Loading from cache: EP03_09.pt
⚡ Loading from cache: EP03_10.pt
⚡ Loading from cache: EP05_02.pt
⚡ Loading from cache: E